In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST
from torchvision.utils import make_grid
from torchmetrics.image.fid import FrechetInceptionDistance

TRAIN = True
SEED, BATCH, FID_N = 42, 128, 2048
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CKPT_DIR, METRIC_DIR = ROOT / "results/checkpoints/custom_dcgan", ROOT / "results/metrics/custom_dcgan"
CKPT_DIR.mkdir(parents=True, exist_ok=True); METRIC_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)

train_raw = FashionMNIST(ROOT / "data", train=True, download=True)
test_raw = FashionMNIST(ROOT / "data", train=False, download=True)
images = torch.cat([train_raw.data, test_raw.data]).unsqueeze(1).float() / 255.0
labels = torch.cat([train_raw.targets, test_raw.targets])

with np.load(ROOT / "results/metrics/fashion_mnist_seed42.npz") as f:
    splits = {k: TensorDataset(images[f[k]], labels[f[k]]) for k in ("train", "val", "test")}

def get_loader(name):
    return DataLoader(splits[name], BATCH, shuffle=(name == "train"), generator=torch.Generator().manual_seed(SEED))

In [ ]:
class DCGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.generator = nn.Sequential(
            nn.Unflatten(1, (100, 1, 1)),
            nn.ConvTranspose2d(100, 64, 7, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh()
        )
        self.discriminator = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 1, 7), nn.Flatten(1)
        )
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
                nn.init.normal_(m.weight, float(isinstance(m, nn.BatchNorm2d)), 0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

class ValExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.head = nn.Linear(128, 10)
    def forward(self, x):
        return self.head(self.net(x))

def frechet_cnn(model, ext, mu_r, sig_r):
    rng = torch.Generator(device=DEVICE).manual_seed(SEED)
    with torch.inference_mode():
        feats = ext.net((model.generator(torch.randn((1024, 100), device=DEVICE, generator=rng)) + 1) / 2).cpu().numpy().astype(np.float64)
    mu_f, sig_f = feats.mean(0), np.cov(feats, rowvar=False) + np.eye(feats.shape[1]) * 1e-6
    vals, vecs = np.linalg.eigh(sig_r @ sig_f)
    covmean = (vecs * np.sqrt(np.clip(vals, 0.0, None))) @ vecs.T
    return float((mu_r - mu_f) @ (mu_r - mu_f) + np.trace(sig_r + sig_f - 2.0 * covmean))

model = DCGAN().to(DEVICE)
chosen = {}

if TRAIN:
    ext = ValExtractor().to(DEVICE)
    opt_e = torch.optim.Adam(ext.parameters(), lr=1e-3)
    for _ in range(3):
        for bx, by in get_loader("train"):
            loss = F.cross_entropy(ext(bx.to(DEVICE)), by.to(DEVICE))
            opt_e.zero_grad(set_to_none=True); loss.backward(); opt_e.step()
    ext.eval()
    with torch.inference_mode():
        rf = torch.cat([ext.net(b.to(DEVICE)).cpu() for b in splits["val"].tensors[0].split(256)]).numpy().astype(np.float64)
    mu_r, sig_r = rf.mean(0), np.cov(rf, rowvar=False) + np.eye(rf.shape[1]) * 1e-6

    opt_g = torch.optim.Adam(model.generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(model.discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    best, hist = float("inf"), []
    for epoch in range(1, 51):
        model.train()
        d_sum, g_sum, n = 0.0, 0.0, 0
        for x, _ in get_loader("train"):
            x = 2 * x.to(DEVICE) - 1
            fake = model.generator(torch.randn(len(x), 100, device=DEVICE))
            r_out, f_out = model.discriminator(torch.cat((x, fake.detach()))).chunk(2)
            loss_d = F.binary_cross_entropy_with_logits(r_out, torch.ones_like(r_out)) + F.binary_cross_entropy_with_logits(f_out, torch.zeros_like(f_out))
            opt_d.zero_grad(set_to_none=True); loss_d.backward(); opt_d.step()
            g_out = model.discriminator(fake)
            loss_g = F.binary_cross_entropy_with_logits(g_out, torch.ones_like(g_out))
            opt_g.zero_grad(set_to_none=True); loss_g.backward(); opt_g.step()
            d_sum += loss_d.item() * len(x); g_sum += loss_g.item() * len(x); n += len(x)
        val_s = None
        if epoch in (15, 30, 50):
            model.eval()
            val_s = frechet_cnn(model, ext, mu_r, sig_r)
            if val_s < best:
                best, chosen["DCGAN"] = val_s, epoch
                best_w = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                torch.save({"epoch": epoch, "val_cnn_fid": val_s, "model_state_dict": best_w}, CKPT_DIR / "dcgan_best.pt")
        hist.append({"epoch": epoch, "d_loss": d_sum / n, "g_loss": g_sum / n, "val_cnn_fid": val_s})
        if epoch % 5 == 0 or epoch == 1:
            print(f"[DCGAN] Epoch {epoch:02d}/50 | D: {d_sum/n:.4f} | G: {g_sum/n:.4f} | Val CNN-FID: {val_s}")
    model.load_state_dict(best_w)
    pd.DataFrame(hist).to_csv(METRIC_DIR / "dcgan_history.csv", index=False)
    (METRIC_DIR / "selected_epoch.json").write_text(json.dumps(chosen, indent=2))
else:
    ckpt = torch.load(CKPT_DIR / "dcgan_best.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"]); model.eval(); chosen["DCGAN"] = ckpt["epoch"]

display(pd.Series(chosen, name="Epoch terpilih").to_frame())

In [ ]:
@torch.inference_mode()
def calc_fid_tm(real_x, fake_x):
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    for b in real_x.split(128): fid.update(b.to(DEVICE).repeat(1, 3, 1, 1), real=True)
    for b in fake_x.split(128): fid.update(b.to(DEVICE).repeat(1, 3, 1, 1), real=False)
    return float(fid.compute())

def calc_latency(fn, runs=30):
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    for _ in range(5): fn()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs): fn()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0 / runs

test_x = splits["test"].tensors[0]
with torch.inference_mode():
    rng = torch.Generator(device=DEVICE).manual_seed(SEED)
    gen_samples = (model.generator(torch.randn((FID_N, 100), device=DEVICE, generator=rng)) + 1) / 2
    fid_test = calc_fid_tm(test_x[:FID_N], gen_samples.cpu())
    lat_ms = calc_latency(lambda: (model.generator(torch.randn((64, 100), device=DEVICE)) + 1) / 2)

    rng_i = torch.Generator(device=DEVICE).manual_seed(SEED + 1)
    z_a, z_b = torch.randn((1, 100), device=DEVICE, generator=rng_i), torch.randn((1, 100), device=DEVICE, generator=rng_i)
    alphas = torch.linspace(0.0, 1.0, 6, device=DEVICE).unsqueeze(1)
    interp = ((model.generator((1 - alphas) * z_a + alphas * z_b) + 1) / 2).cpu()

pd.DataFrame([{"Model": "DCGAN", "Epoch": chosen["DCGAN"], "FID Test (N=2048)": fid_test, "Latensi Batch 64 (ms)": lat_ms}]).to_csv(METRIC_DIR / "dcgan_eval.csv", index=False)
print(f"DCGAN Test FID (TorchMetrics, N={FID_N}): {fid_test:.4f} | Latensi Batch 64: {lat_ms:.3f} ms")

hdf = pd.read_csv(METRIC_DIR / "dcgan_history.csv")
plt.figure(figsize=(6, 3.5)); plt.plot(hdf["epoch"], hdf["d_loss"], label="D Loss"); plt.plot(hdf["epoch"], hdf["g_loss"], label="G Loss"); plt.xlabel("Epoch"); plt.legend(); plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 8)); plt.imshow(make_grid(torch.cat([gen_samples[:48].cpu(), interp]), nrow=6, padding=2).permute(1, 2, 0).numpy()); plt.axis("off"); plt.show()